# Game Simulator
Forecast DAU, payer DAU, and revenue over a 365-day horizon by adjusting UA spend, CPI, retention, and conversion inputs.

In [16]:
# show
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import date

from common_lib.sql import BigQueryConnector
from common_lib.sheets import load_inputs, get_inputs_dir
from common_lib.simulation import (
    SimulationEngine, PlatformInputs,
    save_scenario, load_scenario, list_scenarios,
    save_result, load_result, list_results,
)
from common_lib.widgets import ScenarioPanel

print('Inputs dir:', get_inputs_dir())

Inputs dir: /Users/ivanaguilar/Desktop/DataStuff/gitrepos/testrepo/simulator/config/inputs


## 1. Load baseline actuals from BigQuery

In [ ]:
refresh_data = False  # Set to True to refresh data from BigQuery, False to load from local pickle

In [17]:
# show
bqc = BigQueryConnector()

actuals_params = {'start_date': '2024-01-01'}
cost_info = bqc.print_cost_estimate('./sql/actuals.sql', is_path=True, query_parameters=actuals_params)

This query will process 7.67 GB when run.
Estimated query cost: $0.05


In [ ]:
PLATFORM_MAP = {'AND': 'android', 'IOS': 'ios'}

if refresh_data == True:
    actuals = bqc.get('./sql/actuals.sql', is_path=True, query_parameters=actuals_params)
    pd.to_pickle(actuals, './data/actuals_dau.pkl')
else:
    actuals = pd.read_pickle('./data/actuals_dau.pkl')

actuals['dt'] = pd.to_datetime(actuals['dt'])
actuals['platform'] = actuals['platform'].map(PLATFORM_MAP).fillna(actuals['platform'].str.lower())
actuals = actuals.sort_values('dt')

# Anchor DAU: last observed day per platform
anchor_dau = actuals.sort_values('dt').groupby('platform')['dau'].last().to_dict()
print('Anchor DAU:', anchor_dau)

Anchor DAU: {'android': 35491, 'ios': 50162}


In [19]:
# show
# Fetch live retention and conversion curves
cohort_params = {'start_date': '2024-01-01'}
cost_info = bqc.print_cost_estimate('./sql/retention.sql', is_path=True, query_parameters=cohort_params)
cost_info = bqc.print_cost_estimate('./sql/conversion.sql', is_path=True, query_parameters=cohort_params)

This query will process 4.34 GB when run.
Estimated query cost: $0.03
This query will process 4.36 GB when run.
Estimated query cost: $0.03


In [20]:


if refresh_data == True:
    live_retention  = bqc.get('./sql/retention.sql',  is_path=True, query_parameters=cohort_params)
    live_retention.to_pickle('./data/live_retention.pkl')
    live_conversion = bqc.get('./sql/conversion.sql', is_path=True, query_parameters=cohort_params)
    live_conversion.to_pickle('./data/live_conversion.pkl')
else:
    live_retention  = pd.read_pickle('./data/live_retention.pkl')
    live_conversion = pd.read_pickle('./data/live_conversion.pkl')

live_retention['platform']  = live_retention['platform'].map(PLATFORM_MAP).fillna(live_retention['platform'].str.lower())
live_conversion['platform'] = live_conversion['platform'].map(PLATFORM_MAP).fillna(live_conversion['platform'].str.lower())

## 2. Load CPI and UA spend from local CSVs

In [ ]:
# show
sheet_inputs = load_inputs()

for name, df in sheet_inputs.items():
    print(f'\n--- {name} ---')
    #print(df.to_string(index=False))


--- cpi ---
  month platform  cpi
2026-05      ios  2.5
2026-05  android  1.2
2026-06      ios  2.5
2026-06  android  1.2
2026-07      ios  2.5
2026-07  android  1.2
2026-08      ios  2.5
2026-08  android  1.2
2026-09      ios  2.5
2026-09  android  1.2
2026-10      ios  2.5
2026-10  android  1.2
2026-11      ios  2.5
2026-11  android  1.2
2026-12      ios  2.5
2026-12  android  1.2
2027-01      ios  2.5
2027-01  android  1.2
2027-02      ios  2.5
2027-02  android  1.2
2027-03      ios  2.5
2027-03  android  1.2
2027-04      ios  2.5
2027-04  android  1.2

--- ua_spend ---
  month platform  budget
2026-05      ios   50000
2026-05  android   30000
2026-06      ios   50000
2026-06  android   30000
2026-07      ios   50000
2026-07  android   30000
2026-08      ios   50000
2026-08  android   30000
2026-09      ios   50000
2026-09  android   30000
2026-10      ios   50000
2026-10  android   30000
2026-11      ios   50000
2026-11  android   30000
2026-12      ios   50000
2026-12  android   

## 4. Interactive Scenario Panel

In [22]:
# show
from common_lib.app import prefill_panel, setup_callbacks

engine = SimulationEngine()
panel  = ScenarioPanel(saved_scenarios=list_scenarios())
prefill_panel(panel, actuals, anchor_dau, sheet_inputs)
setup_callbacks(panel, engine, actuals, live_retention=live_retention, live_conversion=live_conversion)
panel.display()

## 5. Plot results

`plot(scenarios, chart)` loads saved results from disk and renders charts.

- **`scenarios`** — a single name or list of names (must have been simulated first)
- **`chart`** — `'all'` (default) · `'dau'` · `'installs'` · `'revenue'` · `'payers'` · `'monthly'`

In [28]:
from common_lib.plots import plot, plot_retention, plot_conversion, configure as configure_plots
from common_lib.simulation import list_results

configure_plots(actuals)

# ── Usage ──────────────────────────────────────────────────────────────────
# plot('base_case')                         # all charts
# plot('base_case', chart='dau')            # DAU only
# plot('base_case', chart='revenue')        # daily revenue
# plot('base_case', chart='monthly')        # monthly bar
# plot(['base_case', 'high_ua'])            # compare scenarios
#
# plot_retention('base_case')               # retention curve from saved scenario
# plot_conversion('base_case')              # conversion curve from saved scenario
# plot_retention(['base_case', 'high_ua'])  # compare retention curves across scenarios
# plot_retention(panel.get_curve_anchors()) # preview current panel state
print('Available results:', list_results())

Available results: ['test3']


In [29]:
plot('test3', chart='dau')

In [30]:
plot('test3', chart='revenue')

In [26]:
plot_conversion('test3')
plot_retention('test3') 

## 6. Summary table

In [27]:
# show
def summary_table(scenarios=None) -> pd.DataFrame:
    """
    Summarise saved simulation results.
    scenarios: list of names, or None to include all saved results.
    """
    names = scenarios if scenarios is not None else list_results()
    rows = []
    for name in names:
        df = load_result(name)
        for platform in ('ios', 'android', 'combined'):
            sub = df[df['platform'] == platform]
            if sub.empty:
                continue
            rows.append({
                'scenario':       name,
                'platform':       platform,
                'avg_dau':        round(sub['dau'].mean()),
                'peak_dau':       round(sub['dau'].max()),
                'total_installs': round(sub['new_installs'].sum()),
                'total_iap_rev':  round(sub['iap_revenue'].sum(), 2),
                'total_ad_rev':   round(sub['ad_revenue'].sum(), 2),
                'total_revenue':  round(sub['total_revenue'].sum(), 2),
            })
    return pd.DataFrame(rows)


summary_table()

,scenario,platform,avg_dau,peak_dau,total_installs,total_iap_rev,total_ad_rev,total_revenue
0,test3,ios,41126,56437,318584,5175829.42,1424553.98,6600383.4
1,test3,android,26132,39898,212389,3367018.68,988167.52,4355186.2
2,test3,combined,67259,96335,530973,8542848.10,2412721.50,10955569.6
